# 03 · Embedding-based near-duplicate scan (Researcher A, Week 2)

MinHash (src/leakage.py) spelling-level near-dup ধরে; এটা paraphrase-level ধরে। LaBSE cosine ≥ threshold → candidate pair; threshold ঠিক করো 100 pair হাতে দেখে।

In [ ]:
# Kaggle setup: Settings -> Accelerator = GPU T4 x2, Internet = ON.
# Option A (recommended): push this repo to your private GitHub and clone it:
#   !git clone https://github.com/<your-user>/bangla-audit.git /kaggle/working/bangla-audit
# Option B: upload the bangla-audit folder as a private Kaggle Dataset named 'bangla-audit', then:
import os, shutil
if not os.path.exists('/kaggle/working/bangla-audit'):
    shutil.copytree('/kaggle/input/bangla-audit', '/kaggle/working/bangla-audit')
os.chdir('/kaggle/working/bangla-audit')
!pip -q install -r requirements.txt
!pip -q install git+https://github.com/csebuetnlp/normalizer
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip -q install sentence-transformers

In [ ]:
import sys, numpy as np, pandas as pd, torch
sys.path.insert(0, 'src')
from common import load_all, normalize
from sentence_transformers import SentenceTransformer
df = load_all()
m = SentenceTransformer('sentence-transformers/LaBSE', device='cuda')
E = m.encode([normalize(t, use_banglabert=False) for t in df.text], batch_size=256,
             convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=True)
test_idx = np.where(df.split.values == 'test')[0]
TH = 0.92  # start here; tune with the manual 100-pair check
rows = []
for s in range(0, len(test_idx), 1024):
    q = test_idx[s:s+1024]
    sims = E[q] @ E.T
    sims[torch.arange(len(q)), torch.tensor(q)] = -1  # drop self
    v, j = sims.topk(5, dim=1)
    for a, (vv, jj) in enumerate(zip(v.cpu().numpy(), j.cpu().numpy())):
        for sc, b in zip(vv, jj):
            if sc >= TH:
                rows.append((df.uid[q[a]], df.uid[b], float(sc), df.label[q[a]], df.label[b]))
P = pd.DataFrame(rows, columns=['uid_test','uid_other','cosine','label_test','label_other'])
P.to_csv('results/leakage/embedding_pairs.csv', index=False)
P.groupby(P.uid_test.str.split(':').str[0]).size()